In [ ]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import statsmodels.api as sm  
from statsmodels.genmod.generalized_linear_model import GLM

In [ ]:
with open('characteristics_with_return.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
def fit_grouped_lasso(X, y, groups, alpha=1.0):  
      
    group_weights = np.array([alpha * len(np.where(groups == i)[0]) for i in np.unique(groups)])  
    weighted_X = X * group_weights[groups]  
      
      
    model = GLM(y, sm.add_constant(weighted_X), family=sm.families.Gaussian())  
    result = model.fit()  
      
     
    coef = result.params[1:] / group_weights[groups]  
    return coef, result

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

In [ ]:
np.random.seed(0)
groups = np.random.randint(0, 5, size=X_train.shape[1])

coef, result = fit_grouped_lasso(X_train, y_train, groups)
X_test_with_const = sm.add_constant(X_test)
    
y_test_pred = result.predict(X_test_with_const)

pred_y_list.extend(y_test_pred)
true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import statsmodels.api as sm  
from statsmodels.genmod.generalized_linear_model import GLM

def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

def fit_grouped_lasso(X, y, groups, alpha=1.0):  
      
    group_weights = np.array([alpha * len(np.where(groups == i)[0]) for i in np.unique(groups)])  
    weighted_X = X * group_weights[groups]  
      
      
    model = GLM(y, sm.add_constant(weighted_X), family=sm.families.Gaussian())  
    result = model.fit()  
      
     
    coef = result.params[1:] / group_weights[groups]  
    return coef, result

r2_differences = {}
feature_names = ['mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade']

with open('characteristics_with_return.pkl', 'rb') as f:
    data = pickle.load(f)

start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

for name in feature_names:

    pred_y_list = []
    true_y_list = []

    selected_features = [name]
    
    original_r2 = 0.4369389045826891

    for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
        """
        定义训练集和测试集的日期范围
        """
        # 训练集的开始日期
        start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
        # 训练集的结束日期
        end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)

        # 验证集的开始日期
        start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
        # 验证集的结束日期
        end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)

        # 测试集的开始日期
        start_date_test = pd.Timestamp(year=year, month=1, day=1)
        # 测试集的结束日期
        end_date_test = pd.Timestamp(year=year, month=12, day=31)

        """
        打印日期范围信息
        """
        print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
        print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

        print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
        print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')

        print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
        print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

        """
        选择训练集和测试集的数据
        """
        train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
        test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

        """
        提取指定的特征和目标变量
        """
        X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
        y_train = train_data['RET'].values  
        X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_test = test_data['RET'].values

        X_train_copy = X_train.copy()
        X_test_copy = X_test.copy()

        X_train_copy[:, selected_features.index(name)] = 0
        X_test_copy[:, selected_features.index(name)] = 0

        np.random.seed(0)
        groups = np.random.randint(0, 10, size=X_train_copy.shape[1])

        coef, result = fit_grouped_lasso(X_train_copy, y_train, groups)
        X_test_with_const = sm.add_constant(X_test_copy)
    
        y_test_pred = result.predict(X_test_with_const)

        pred_y_list.extend(y_test_pred)
        true_y_list.extend(y_test)

    new_r2 = R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100
    r2_difference = original_r2 - new_r2
    r2_differences[name] = r2_difference

    print("每个特征设为 0 时的 R2 差值:")
    for feature, difference in r2_differences.items():
        print(f"{feature}: {difference}")